# 09 — Seasonality Features: Dummies, Fourier Terms & Beyond

**Companion notebook to Article 9 of the *Time Series Feature Engineering* series.**

📖 Read the article: [Medium](https://medium.com/@asidd24) · [LinkedIn Newsletter](https://www.linkedin.com/newsletters/time-series-engineered-7485187061080137728/)

---

This notebook walks through every code example and figure from Article 9, demonstrating:

1. **The ordinal trap** — why raw integer calendar columns hurt accuracy
2. **Dummy encoding** — the correct encoding for short, irregular seasonal cycles
3. **Cyclical sin/cos encoding** — fixes adjacency but not shape
4. **Fourier terms** — the right encoding for long seasonal periods
5. **Head-to-head benchmark** — which encoding actually moves forecast accuracy
6. **The `add_seasonal_features` recipe** — the 19-column practical default

## 1. Setup & Environment

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.linear_model import LinearRegression

from src.features import (
    add_cyclical_encoding,
    add_datetime_features,
    add_fourier_terms,
    add_lag_features,
    add_rolling_features,
    add_seasonal_features,
)
from src.metrics import wmape

sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (12, 4),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
})
np.random.seed(42)

BLUE, GREEN, RED, GRAY, ORANGE, PURPLE, DARK = (
    '#1f77b4', '#2ca02c', '#d62728', '#9aa0a6', '#ff7f0e', '#9467bd', '#333333'
)
DAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
MONTHS = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
print("✅ Setup complete")

## 2. Data: A Deliberately Non-Sinusoidal Series

The `generate_daily_sales` function in `src/data.py` uses pure sine waves for both
weekly and yearly seasonality. That is convenient for trend/lag articles, but it
would **rig** this comparison: a single sin/cos pair reproduces a pure sine exactly.

Real business seasonality is not a sine wave. So we build one that isn't either:
- **Weekly**: a sharp, irregular profile (Friday/Saturday peak, Sunday collapse)
- **Yearly**: a smooth annual wave + a Gaussian December holiday spike + a summer dip

In [ ]:
# Weekly profile: Mon..Sun — sharp, irregular, no smooth curve connects them
DOW_EFFECT = np.array([-2.0, -3.0, -1.5, 0.5, 12.0, 18.0, -24.0])

def yearly_effect(doy: np.ndarray) -> np.ndarray:
    """Smooth annual swing + sharp December peak + summer dip."""
    base = 12.0 * np.sin(2 * np.pi * (doy - 80) / 365.25)
    holiday = 28.0 * np.exp(-0.5 * ((doy - 345) / 12.0) ** 2)
    summer_dip = -10.0 * np.exp(-0.5 * ((doy - 200) / 18.0) ** 2)
    return base + holiday + summer_dip

def make_series(n_days=1095, start='2022-01-01', seed=7,
                noise_std=6.0, trend_slope=0.03, base_level=120.0):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, periods=n_days, freq='D')
    t = np.arange(n_days)
    values = (base_level
              + trend_slope * t
              + DOW_EFFECT[dates.dayofweek.values]
              + yearly_effect(dates.dayofyear.values)
              + rng.normal(0, noise_std, n_days))
    return pd.Series(values, index=dates, name='sales')

sales = make_series()
print(f"Series: {len(sales)} days, {sales.index.min().date()} → {sales.index.max().date()}")
print(f"Noise floor (irreducible RMSE): {6.0:.1f}")

## 3. Figure 1 — Seasonality Is Never One Cycle

A real business series almost never sits on one calendar cycle at a time.
The weekly and yearly profiles are running simultaneously, on completely different
timescales, with completely different shapes.

In [ ]:
fig = plt.figure(figsize=(14, 7.5))
gs = fig.add_gridspec(2, 2, height_ratios=[1.25, 1], hspace=0.42, wspace=0.22)

# Top: full series with weekly cycle removed
ax0 = fig.add_subplot(gs[0, :])
ax0.plot(sales.index, sales.values, color=BLUE, lw=0.8, alpha=0.75, label='Daily sales')
ax0.plot(sales.index, sales.rolling(28, center=True).mean(),
         color=DARK, lw=2.2, label='28-day centred mean (weekly cycle removed)')
ax0.set_title('Two Seasonal Cycles Are Stacked on Top of Each Other', loc='left')
ax0.set_ylabel('Sales')
ax0.legend(loc='upper left')

# Bottom left: weekly profile
weekly_resid = sales - sales.rolling(7, center=True).mean()
weekly_profile = weekly_resid.groupby(sales.index.dayofweek).mean()

ax1 = fig.add_subplot(gs[1, 0])
colors_bar = [GREEN if v >= 0 else RED for v in weekly_profile.values]
ax1.bar(DAYS, weekly_profile.values, color=colors_bar, alpha=0.85)
ax1.axhline(0, color=DARK, lw=1)
ax1.set_title('Weekly profile — sharp, irregular', loc='left', fontsize=13)
ax1.set_ylabel('Deviation from week mean')

# Bottom right: yearly profile
t_arr = np.arange(len(sales)).reshape(-1, 1)
detrended = sales.values - LinearRegression().fit(t_arr, sales.values).predict(t_arr)
monthly = pd.Series(detrended, index=sales.index).groupby(sales.index.month).mean()

ax2 = fig.add_subplot(gs[1, 1])
ax2.plot(MONTHS, monthly.values, color=PURPLE, lw=2.4, marker='o', ms=6)
ax2.axhline(0, color=DARK, lw=1)
ax2.set_title('Yearly profile — smooth, with a December spike', loc='left', fontsize=13)
ax2.set_ylabel('Detrended deviation')

plt.tight_layout()
plt.show()

## 4. The Ordinal Trap

The most common seasonality bug in production forecasting code:

```python
df['month'] = df.index.month        # 1, 2, 3, ..., 12
df['dayofweek'] = df.index.dayofweek  # 0, 1, 2, ..., 6
```

These tell the model December and January are **11 units apart** — the largest
distance on the calendar — when they are adjacent. The sin/cos fix maps the
calendar onto a **circle**, preserving adjacency everywhere.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

# Left: integer encoding on a number line
ax = axes[0]
months = np.arange(1, 13)
ax.scatter(months, np.zeros_like(months), s=190, color=BLUE, zorder=3)
for m, name in zip(months, MONTHS):
    ax.annotate(name, (m, 0), textcoords='offset points', xytext=(0, -26),
                ha='center', fontsize=10)
    ax.annotate(str(m), (m, 0), textcoords='offset points', xytext=(0, 16),
                ha='center', fontsize=9, color=GRAY)

ax.annotate('', xy=(12, 0.38), xytext=(1, 0.38),
            arrowprops=dict(arrowstyle='<->', color=RED, lw=2.4))
ax.text(6.5, 0.46, 'Dec → Jan reads as 11 units apart', ha='center',
        color=RED, fontsize=12, fontweight='bold')
ax.annotate('', xy=(7, -0.42), xytext=(6, -0.42),
            arrowprops=dict(arrowstyle='<->', color=GREEN, lw=2.4))
ax.text(6.5, -0.56, 'Jun → Jul reads as 1 unit apart', ha='center',
        color=GREEN, fontsize=12, fontweight='bold')
ax.set_ylim(-0.8, 0.7); ax.set_xlim(0.2, 12.8)
ax.set_yticks([]); ax.set_xticks([]); ax.grid(False)
ax.set_title('Integer encoding — December and January sit at opposite ends',
             loc='left', fontsize=13)

# Right: sin/cos encoding on the unit circle
ax = axes[1]
ang = 2 * np.pi * (months - 1) / 12
x, y = np.sin(ang), np.cos(ang)
circle = plt.Circle((0, 0), 1, fill=False, color=GRAY, lw=1.4, ls='--')
ax.add_patch(circle)
ax.scatter(x, y, s=190, color=BLUE, zorder=3)
for xi, yi, name in zip(x, y, MONTHS):
    ax.annotate(name, (xi * 1.17, yi * 1.17), ha='center', va='center', fontsize=10)

d_dec_jan = float(np.hypot(x[11] - x[0], y[11] - y[0]))
d_jun_jul = float(np.hypot(x[6] - x[5], y[6] - y[5]))
ax.plot([x[11], x[0]], [y[11], y[0]], color=RED, lw=3, zorder=4)
ax.plot([x[5], x[6]], [y[5], y[6]], color=GREEN, lw=3, zorder=4)
ax.text(0, -1.42, f'Dec→Jan distance = {d_dec_jan:.3f}   ·   '
                   f'Jun→Jul distance = {d_jun_jul:.3f}   →   identical',
        ha='center', fontsize=12, fontweight='bold', color=DARK)
ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.65, 1.5)
ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
ax.set_title('sin/cos encoding — the calendar closes into a circle',
             loc='left', fontsize=13)

plt.tight_layout()
plt.show()
print(f"Dec→Jan Euclidean distance: {d_dec_jan:.4f}")
print(f"Jun→Jul Euclidean distance: {d_jun_jul:.4f}")

### Using `add_cyclical_encoding`

The fix is two columns per cycle:

In [ ]:
df = sales.to_frame()
df = add_datetime_features(df, features=('dayofweek', 'month'))
df = add_cyclical_encoding(df, 'dayofweek', period=7)
df = add_cyclical_encoding(df, 'month', period=12)

print("Cyclical columns added:")
print(df[['dayofweek', 'dayofweek_sin', 'dayofweek_cos',
          'month', 'month_sin', 'month_cos']].head(10))

## 5. Figure 3 — A Single sin/cos Pair Cannot Represent a Sharp Weekly Shape

One-hot dummies make **no assumption about shape** — each position gets its own
free parameter. A single sin/cos pair is **one smooth wave**. Let's compare:

In [ ]:
resid = (sales - sales.rolling(7, center=True).mean()).dropna()
dow = resid.index.dayofweek.values
truth = DOW_EFFECT - DOW_EFFECT.mean()

# (a) 7 dummies == group means
dummies = pd.Series(resid.values).groupby(dow).mean().values

# (b) single sin/cos pair
X1 = np.column_stack([np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7)])
m1 = LinearRegression().fit(X1, resid.values)
g = np.arange(7)
cyc = m1.predict(np.column_stack([np.sin(2 * np.pi * g / 7), np.cos(2 * np.pi * g / 7)]))

# (c) 3 Fourier pairs at period 7
def fourier_block(d, k_max, period=7):
    cols = []
    for k in range(1, k_max + 1):
        cols += [np.sin(2 * np.pi * k * d / period), np.cos(2 * np.pi * k * d / period)]
    return np.column_stack(cols)

m3 = LinearRegression().fit(fourier_block(dow, 3), resid.values)
fou3 = m3.predict(fourier_block(g, 3))

fig, ax = plt.subplots(figsize=(13, 5.6))
ax.plot(DAYS, truth, color=DARK, lw=3.4, marker='o', ms=9, label='True weekly effect', zorder=5)
ax.plot(DAYS, dummies, color=GREEN, lw=2.4, marker='s', ms=8, ls='--',
        label='7 day-of-week dummies (6 free params)')
ax.plot(DAYS, cyc, color=RED, lw=2.4, marker='^', ms=8, ls='--',
        label='1 sin/cos pair (2 params)')
ax.plot(DAYS, fou3, color=ORANGE, lw=2.4, marker='v', ms=8, ls=':',
        label='3 Fourier pairs, period 7 (6 params)')
ax.axhline(0, color=GRAY, lw=1)
ax.set_ylabel('Effect on sales')
ax.set_title("A Single sin/cos Pair Cannot Represent a Sharp Weekly Shape", loc='left')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

e_dum = np.abs(dummies - truth).max()
e_cyc = np.abs(cyc - truth).max()
e_f3  = np.abs(fou3 - truth).max()
print(f"Max abs error — dummies: {e_dum:.2f}, 1 sin/cos pair: {e_cyc:.2f}, "
      f"3 Fourier pairs: {e_f3:.2f}")

## 6. Figure 4 — How Many Fourier Pairs Does a Yearly Cycle Need?

K controls the smoothness: each pair adds a higher-frequency harmonic.
Too low → flatten real events. Too high → ring around sharp peaks.

In [ ]:
# Isolate the yearly component
t_arr = np.arange(len(sales)).reshape(-1, 1)
detr = sales.values - LinearRegression().fit(t_arr, sales.values).predict(t_arr)
s = pd.Series(detr, index=sales.index)
dow_means = s.groupby(s.index.dayofweek).transform('mean')
yc = s - dow_means

doy = yc.index.dayofyear.values
grid = np.arange(1, 366)
truth_yearly = yearly_effect(grid) - yearly_effect(grid).mean()

def fourier_block_yearly(d, k_max, period=365.25):
    cols = []
    for k in range(1, k_max + 1):
        cols += [np.sin(2 * np.pi * k * d / period), np.cos(2 * np.pi * k * d / period)]
    return np.column_stack(cols)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(grid, truth_yearly, color=DARK, lw=3.6, label='True yearly effect', zorder=6)

colours = {1: RED, 2: ORANGE, 3: PURPLE, 6: GREEN, 10: BLUE}
for k, c in colours.items():
    m = LinearRegression().fit(fourier_block_yearly(doy, k), yc.values)
    pred = m.predict(fourier_block_yearly(grid, k))
    pred = pred - pred.mean()
    rmse = float(np.sqrt(np.mean((pred - truth_yearly) ** 2)))
    ax.plot(grid, pred, color=c, lw=2.0, ls='--',
            label=f'K = {k:>2} ({2*k:>2} columns) — RMSE {rmse:.2f}')
    print(f"  K={k:>2} ({2*k:>2} cols): RMSE={rmse:.3f}")

ax.axvline(345, color=GRAY, lw=1.2, ls=':')
ax.annotate('December holiday peak', xy=(345, truth_yearly.max()),
            xytext=(232, truth_yearly.max() + 6), fontsize=11, color=DARK,
            arrowprops=dict(arrowstyle='->', color=DARK, lw=1.4))
ax.set_xlabel('Day of year')
ax.set_ylabel('Effect on sales')
ax.set_title('More Fourier Pairs = More Seasonal Detail (K Controls the Smoothness)', loc='left')
ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

## 7. Figure 5 — Day-of-Year Dummies Overfit; Fourier Generalises

365 dummies give each parameter ~2-3 observations. Fourier terms use all
observations to estimate a compact set of parameters.

In [ ]:
train_yc = yc[yc.index.year < 2024]      # 2 years
test_yc  = yc[yc.index.year == 2024]     # held-out year

grid = np.arange(1, 366)
truth_yearly = yearly_effect(grid) - yearly_effect(grid).mean()

# (a) Day-of-year dummies
dummy_map = train_yc.groupby(train_yc.index.dayofyear).mean()
dummy_curve = np.array([dummy_map.get(d, np.nan) for d in grid], dtype=float)
dummy_curve = pd.Series(dummy_curve).interpolate(limit_direction='both').values
dummy_curve = dummy_curve - np.nanmean(dummy_curve)

# (b) 6 Fourier pairs
m = LinearRegression().fit(fourier_block_yearly(train_yc.index.dayofyear.values, 6), train_yc.values)
fourier_curve = m.predict(fourier_block_yearly(grid, 6))
fourier_curve = fourier_curve - fourier_curve.mean()

# Held-out-year error
td = test_yc.index.dayofyear.values
dummy_pred = np.array([dummy_map.get(d, dummy_map.mean()) for d in td], dtype=float)
fourier_pred = m.predict(fourier_block_yearly(td, 6))
rmse_dummy   = float(np.sqrt(np.mean((dummy_pred - test_yc.values) ** 2)))
rmse_fourier = float(np.sqrt(np.mean((fourier_pred - test_yc.values) ** 2)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6), sharey=True)

ax = axes[0]
ax.plot(grid, dummy_curve, color=RED, lw=1.3, alpha=0.9, label='Fitted (365 dummies)')
ax.plot(grid, truth_yearly, color=DARK, lw=3, label='True yearly effect')
ax.set_title('365 day-of-year dummies — chases noise', loc='left', fontsize=13)
ax.set_xlabel('Day of year'); ax.set_ylabel('Effect on sales')
ax.legend(loc='upper left', fontsize=10)
ax.text(0.02, 0.03, f'365 parameters · ~2 obs each\nheld-out-year RMSE = {rmse_dummy:.2f}',
        transform=ax.transAxes, fontsize=11, color=RED, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', fc='white', ec=RED, alpha=0.9))

ax = axes[1]
ax.plot(grid, fourier_curve, color=GREEN, lw=2.6, label='Fitted (6 Fourier pairs)')
ax.plot(grid, truth_yearly, color=DARK, lw=3, label='True yearly effect')
ax.set_title('6 Fourier pairs — recovers the shape', loc='left', fontsize=13)
ax.set_xlabel('Day of year')
ax.legend(loc='upper left', fontsize=10)
ax.text(0.02, 0.03, f'12 parameters · all 730 obs\nheld-out-year RMSE = {rmse_fourier:.2f}',
        transform=ax.transAxes, fontsize=11, color=GREEN, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', fc='white', ec=GREEN, alpha=0.9))

fig.suptitle('Long Seasonal Periods Break Dummy Encoding', x=0.125, ha='left',
             fontsize=15, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

print(f"Held-out-year RMSE — 365 dummies: {rmse_dummy:.2f}, 6 Fourier pairs: {rmse_fourier:.2f}")

## 8. Figure 6 — The Head-to-Head: Which Encoding Actually Wins?

LightGBM, 7-day-ahead forecasting. Same base features (lags 7/14/21/28,
rolling mean/std over 7 and 28 days, linear t_index). **Only** the seasonal
encoding changes.

In [ ]:
H = 7  # forecast horizon
df = sales.to_frame()

df = add_lag_features(df, 'sales', lags=[7, 14, 21, 28])
df = add_rolling_features(df, 'sales', windows=[7, 28], functions=('mean', 'std'),
                          min_horizon=H)
df['t_index'] = np.arange(len(df))
df = add_datetime_features(df, features=('dayofweek', 'month'))

# Encoding blocks
dow_dummies = pd.get_dummies(df['dayofweek'], prefix='dow').astype(int)
df = pd.concat([df, dow_dummies], axis=1)
df = add_cyclical_encoding(df, 'dayofweek', period=7)
df = add_cyclical_encoding(df, 'month', period=12)
df = add_fourier_terms(df, period=365.25, n_terms=6, prefix='fy')

base = (['sales_lag_7', 'sales_lag_14', 'sales_lag_21', 'sales_lag_28']
        + [c for c in df.columns if c.startswith('sales_roll_')]
        + ['t_index'])
fourier_cols = [c for c in df.columns if c.startswith('fy_')]
dummy_cols = list(dow_dummies.columns)

feature_sets = {
    'Lags + trend only\n(no calendar)': base,
    '+ integer dayofweek & month': base + ['dayofweek', 'month'],
    '+ day-of-week dummies': base + dummy_cols,
    '+ cyclical sin/cos': base + ['dayofweek_sin', 'dayofweek_cos',
                                   'month_sin', 'month_cos'],
    '+ Fourier yearly (K=6)': base + fourier_cols,
    '+ dummies & Fourier (K=6)': base + dummy_cols + fourier_cols,
}

model_df = df.dropna().copy()
split = model_df.index < '2024-01-01'
train, test = model_df[split], model_df[~split]
print(f"Train: {len(train)} days, Test: {len(test)} days (7-day-ahead)")

results = {}
for name, cols in feature_sets.items():
    model = lgb.LGBMRegressor(
        n_estimators=400, learning_rate=0.05, num_leaves=31,
        random_state=42, verbose=-1,
    )
    model.fit(train[cols], train['sales'])
    pred = model.predict(test[cols])
    results[name] = wmape(test['sales'].values, pred) * 100

order = sorted(results, key=results.get, reverse=True)
vals = [results[k] for k in order]
best = min(results, key=results.get)
colours_bar = [GREEN if k == best else (RED if k == max(results, key=results.get) else BLUE)
               for k in order]

fig, ax = plt.subplots(figsize=(13, 6.4))
bars = ax.barh(range(len(order)), vals, color=colours_bar, alpha=0.88)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=11)
ax.set_xlabel('WMAPE (%) on held-out year — lower is better')
ax.set_title('Seasonal Encoding Changes 7-Day-Ahead Accuracy Materially', loc='left')
ax.set_xlim(0, max(vals) * 1.18)

baseline_key = 'Lags + trend only\n(no calendar)'
baseline_val = results[baseline_key]
for b, v, k in zip(bars, vals, order):
    if k == baseline_key:
        label = f'{v:.2f}%   (baseline)'
    else:
        delta = (baseline_val - v) / baseline_val * 100
        label = f'{v:.2f}%   ({delta:+.0f}% vs baseline)'
    ax.text(v + max(vals) * 0.012, b.get_y() + b.get_height() / 2, label,
            va='center', fontsize=11, fontweight='bold')
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.show()

print("\nWMAPE (%) and change vs no-calendar baseline:")
for k in order:
    d = (baseline_val - results[k]) / baseline_val * 100
    print(f"  {k.replace(chr(10), ' '):<34} {results[k]:.3f}   ({d:+.1f}%)")

## 9. The `add_seasonal_features` Recipe

The complete seasonal feature set for a daily series — 19 columns total:
- 7 day-of-week dummies (short cycle, irregular shape)
- 12 yearly Fourier columns (long cycle, smooth shape)

Build this on the **full contiguous frame**, before any train/test split.

In [ ]:
# The add_seasonal_features convenience function from src/features.py
seasonal_df = add_seasonal_features(sales.to_frame(), yearly_k=6)

seasonal_cols = [c for c in seasonal_df.columns if c != 'sales']
print(f"Seasonal columns added: {len(seasonal_cols)}")
for c in seasonal_cols:
    print(f"  {c}")

## 10. Decision Framework — Which Encoding When

| Situation | Encoding |
|---|---|
| Short period (≤12), irregular shape, many obs per position | **One-hot dummies** |
| Short period, smooth shape, few obs per position | **1-2 Fourier pairs** |
| Long period (52, 365.25), daily or weekly data | **Fourier terms, K=3-10** |
| Non-integer period (365.25 days, 30.4-day month) | **Fourier terms** — dummies cannot represent it |
| Multiple seasonalities at once | **Dummies for the short cycle + Fourier for the long one** |
| Raw integer `month` / `dayofweek` column | **Delete it** — or at minimum replace with sin/cos |
| Tree model whose lags already span the period | Add it, but **measure** — it may be redundant |
| Linear model, any seasonality | **Always encode explicitly** — it has no other route in |

### The Practical Progression for a Daily Business Series

1. Day-of-week dummies (7 columns) — cheap, safe, usually a small win
2. Yearly Fourier terms at K=6 (12 columns) — usually the big win
3. Tune K by held-out error: try 3, 6, 10
4. Add month dummies only if a month-boundary effect exists that the yearly Fourier terms miss
5. Delete every raw integer calendar column you find

## 11. Leakage Rules for Seasonality Features

✅ **Inherently leakage-free.** Calendar encodings are deterministic functions of
the timestamp — they use no observed target values.

⚠️ **Three caveats:**

1. **Build Fourier terms on the full contiguous frame, before splitting.** `add_fourier_terms`
   indexes from `t = np.arange(len(df))` starting at the first row. Separate builds on train
   and test restart at phase zero, silently misaligning every seasonal column.

2. **Seasonal indices *estimated from data* are not calendar features.** A day-of-week
   multiplier computed as a group mean over the target requires walk-forward discipline.

3. **Holiday/event flags are only safe if the calendar is known in advance.** Fixed-date
   holidays are fine. Back-filled flags ("promotion was running") are future-information leaks.

## 12. Key Takeaways

- Real series carry **multiple seasonal cycles at once**, with different periods *and* shapes
- **Raw integer calendar columns assert false distances** — measured *worse than no calendar features at all*
- **Cyclical sin/cos fixes adjacency, not shape** — nearly worthless for irregular short cycles
- **Dummies** are right for short periods with many observations per position
- **Fourier terms** are right for long periods — 12 columns beat 365 day-of-year dummies on held-out error
- **K is a smoothness dial** — K=6 is a good default for yearly-on-daily
- The biggest win was **yearly Fourier terms: ~31% WMAPE reduction** — day-of-week dummies gave only ~4%
- **Seasonal value scales with distance beyond your longest lag** — encode the cycles your lags can't reach
- Calendar encodings are **leakage-free by construction**, but mind the three caveats above

---

📖 **Next:** Article 10 — *Datetime & Categorical Features: The Last Mile of Time Series FE*